<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.2-power-grid-stability-prediction/Ex12.2_02_dense_baseline_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.2 · Notebook 02 — The Dense Baseline

**Paired with L12.2 · Prediction of Power Grid Stability**

The obvious thing to do with the dataset from notebook 01 is to flatten it.
Six buses times six channels is a 36-vector; staple a six-bit one-hot on the
end to say which line is out; feed the 42 numbers to a fully connected network
and regress the critical clearing time.

It works. That is the point of this notebook: **the dense baseline is good**,
and you have to see how good before the argument for a graph network means
anything.

It is also, in two specific ways, not a model you could deploy — and this
notebook measures both of them so that notebook 03 has something to improve on
rather than a story to tell.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.2-power-grid-stability-prediction/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
import os
os.makedirs(pb.RESULTS, exist_ok=True)

data = pb.build_dataset(n_ops=180, seed=12)
cases = pb.contingencies()

X, A, onehot = data["X"], data["A"], data["onehot"]
cct, op_id, cont_id = data["cct"], data["op_id"], data["cont_id"]

n_train_ops = 144
train = op_id < n_train_ops
test = ~train

Z = pb.dense_inputs(X, onehot)          # (n, 6*6 + 6) = (n, 42)
check_shape("dense inputs", Z, (1080, 42))
print()
print(f"  train {train.sum()} cases / test {test.sum()} cases,"
      f" split on the dispatch")
print(f"  target: CCT in seconds, {cct.min():.3f} .. {cct.max():.3f}")

**What you should see.** A PASS on `(1080, 42)`, and 864 / 216.

Forty-two numbers. Look at what `dense_inputs` did:

```python
np.concatenate([X.reshape(X.shape[0], -1), onehot], axis=1)
```

Column 13 of that vector means "reactive power at bus 2" **only because
somebody listed the buses in that order**. The network's first weight matrix is
indexed by position, so the bus ordering is now baked into the parameters. Hold
that thought until notebook 03 section 6.

---

## 1 · Scaling, and why the target is in milliseconds

Two housekeeping decisions that change the numbers you will report.

**Inputs are standardised** using the *training* mean and standard deviation
only. Using the whole dataset's statistics leaks the test set into the
preprocessing — a small leak, and the kind that is impossible to find later.

**The target is scaled to milliseconds and divided by 100.** A CCT in seconds
is a number around 0.2, and a `tanh` network with Xavier initialisation
produces outputs of order 1. Asking it to shrink everything by a factor of five
before it can start fitting wastes the first few hundred steps. Report in
milliseconds throughout; that is the unit a protection engineer thinks in.

In [ ]:
mu, sd = Z[train].mean(axis=0), Z[train].std(axis=0)
live = sd > 1e-12                       # columns that vary at all
sd = np.where(live, sd, 1.0)            # constant columns: leave them alone
Zs = (Z - mu) / sd

Y_SCALE = 0.1                           # seconds -> units of 100 ms
y = cct / Y_SCALE

print(f"  columns that vary in training : {int(live.sum())} of {Z.shape[1]}")
print(f"  columns that are constant     : {int((~live).sum())}")
print(f"  standardised, varying columns : mean {Zs[train][:, live].mean():+.2e},"
      f" std {Zs[train][:, live].std():.4f}")
print(f"  scaled target                 : {y.min():.3f} .. {y.max():.3f}")

**What you should see.** **29 of the 42 columns vary**; the other 13 are
constant across the whole training set and are left alone rather than divided by
a standard deviation of zero. On the 29 that vary, the standardised training
mean is of order `1e-14` and the standard deviation is exactly 1.0000. The
scaled target runs from 0.557 to 5.000.

The 13 dead columns are worth a look: they are `P` and `Q` at the slack bus,
which are zero by definition of a slack bus, the `is_machine` channel at every
bus, which is structural, and the one-hot bit for the islanding branch, which
never fires. A dense model spends 13 input weights per hidden unit on them and
learns to ignore them. That is not a disaster; it is a small, concrete example
of what "the representation carries no structure" costs.

---

## 2 · The baseline you have to beat

Before a network, a straight line. Ridge regression on the same 42 inputs takes
a millisecond to fit and is the honest floor: if your network cannot beat this,
it has learned nothing that a linear model did not already have.

This cell is complete — run it and keep the number.

In [ ]:
def ridge_fit(Ztr, ytr, lam=1e-6):
    "Least squares with a small ridge, on standardised inputs plus a bias."
    Ztr = np.concatenate([Ztr, np.ones((len(Ztr), 1))], axis=1)
    G = Ztr.T @ Ztr + lam * len(Ztr) * np.eye(Ztr.shape[1])
    w = np.linalg.solve(G, Ztr.T @ ytr)
    return lambda Zq: np.concatenate([Zq, np.ones((len(Zq), 1))], axis=1) @ w


linear = ridge_fit(Zs[train], cct[train])
lin_train = linear(Zs[train])
lin_test = linear(Zs[test])

print(f"  linear, train MAE : {np.abs(cct[train]-lin_train).mean()*1e3:6.2f} ms")
print(f"  linear, test  MAE : {np.abs(cct[test]-lin_test).mean()*1e3:6.2f} ms")
print(f"  linear, test worst: {np.abs(cct[test]-lin_test).max()*1e3:6.1f} ms")
c_lin = pb.confusion(cct[test], lin_test)
print(f"  at {pb.PROTECTION_TIME*1e3:.0f} ms: missed {c_lin['missed']},"
      f" false alarms {c_lin['alarms']},"
      f" recall {c_lin['recall']:.3f}, accuracy {c_lin['accuracy']:.3f}")
print()
print(f"  for scale: predicting the training mean gives a test MAE of "
      f"{np.abs(cct[test]-cct[train].mean()).mean()*1e3:.1f} ms")

**What you should see.**

```
  linear, train MAE :  21.49 ms
  linear, test  MAE :  20.72 ms
  linear, test worst:  84.0 ms
  at 140 ms: missed 0, false alarms 5, recall 1.000, accuracy 0.977
```

against **108.2 ms** for predicting the mean.

That is a strong baseline and it should make you suspicious of easy praise
later. Twenty milliseconds is one cycle at 50 Hz. A linear model on 42 numbers
already catches every insecure case in the test split at the cost of five false
alarms.

Two reasons it does so well. The CCT depends smoothly and nearly monotonically
on the machine loading, which is one of the 42 inputs; and the one-hot tells it
exactly which of six regimes it is in, so it is really fitting six almost-linear
functions at once. Both of those advantages are about to become liabilities.

---

## 3 · The dense network

### Your turn

In [ ]:
# TODO 1 --- a dense regressor on the flattened state ---------------------------------------------------
# Three `...` to replace:
#   line 1  ->  MLP(n_in=42, n_out=1, n_hidden=64, n_layers=4)
#   line 2  ->  mse(dense(Ztr) - ytr)                              ytr is already a column
#   line 3  ->  to_numpy(dense(Zte)).ravel() * Y_SCALE              back to seconds
set_seed(88)
dense = ...                                       # <- MLP(n_in=42, n_out=1, n_hidden=64, n_layers=4)
print("parameters:", parameter_count(dense))

Ztr = to_tensor(Zs[train]);  ytr = to_tensor(y[train])
Zte = to_tensor(Zs[test]);   yte = to_tensor(y[test])

def loss_dense():
    return ...                                    # <- mse(dense(Ztr) - ytr)

history_dense = train_two_stage(dense, loss_dense,
                                adam_steps=3000, lbfgs_steps=120,
                                lr=3e-3, report_every=500)

with torch.no_grad():
    pred_dense_train = to_numpy(dense(Ztr)).ravel() * Y_SCALE
    pred_dense_test  = ...                        # <- to_numpy(dense(Zte)).ravel() * Y_SCALE
# ------------------------------------------------------------------------------

In [ ]:
plot_curves(history_dense, title="dense baseline on the flattened state")
plt.show()

print(f"  parameters        : {parameter_count(dense):,}")
print(f"  final loss        : {history_dense['lbfgs'][-1]:.4e}")
print(f"  train MAE         : {np.abs(cct[train]-pred_dense_train).mean()*1e3:6.2f} ms")
print(f"  test  MAE         : {np.abs(cct[test]-pred_dense_test).mean()*1e3:6.2f} ms")
print(f"  test  worst       : {np.abs(cct[test]-pred_dense_test).max()*1e3:6.1f} ms")
print(f"  linear baseline   : {np.abs(cct[test]-lin_test).mean()*1e3:6.2f} ms")

**What you should see.** A loss curve that falls steeply under Adam and then
drops again when L-BFGS takes over — the second drop is the signature of the
two-stage schedule and it is worth two orders of magnitude in Ex_06's
measurements.

Whether your test MAE beats the linear baseline's **20.72 ms** is the result of
this section. Report the number you got, not the one you hoped for. If the
network is *worse* than the line, that is a finding: say so, and check the
obvious cause first — a network that is over-parameterised for 864 examples and
has memorised them.

---

## 4 · Two currencies: milliseconds, and the decision

A regression error in milliseconds is one way to score this. It is not the way
the model will be used. A screening tool makes a **decision**, and the decision
is scored by the two ways it can be wrong.

In [ ]:
uncensored = cct < pb.CCT_MAX - 1e-9
near = np.abs(cct - pb.PROTECTION_TIME) < 0.05

print("  regression, on the test split")
print(f"    all test cases            {np.abs(cct[test]-pred_dense_test).mean()*1e3:6.2f} ms"
      f"   ({test.sum()} cases)")
m = test & uncensored
print(f"    uncensored only           {np.abs(cct[m]-pred_dense_test[uncensored[test]]).mean()*1e3:6.2f} ms"
      f"   ({m.sum()} cases)")
m = test & near
print(f"    within 50 ms of threshold {np.abs(cct[m]-pred_dense_test[near[test]]).mean()*1e3:6.2f} ms"
      f"   ({m.sum()} cases)")

print()
print("  decision, at the protection time")
c_dense = pb.confusion(cct[test], pred_dense_test)
for k in ("tp", "fp", "fn", "tn"):
    print(f"    {k:>3s}  {c_dense[k]:4d}")
print(f"    missed insecure cases  {c_dense['missed']:4d}"
      f"   <- each of these is a blackout the screen did not see")
print(f"    false alarms           {c_dense['alarms']:4d}"
      f"   <- each of these is an engineer's afternoon")
print(f"    recall                 {c_dense['recall']:.3f}")
print(f"    accuracy               {c_dense['accuracy']:.3f}"
      f"   (always-secure scores {(cct[test] > pb.PROTECTION_TIME).mean():.3f})")

pb.plot_parity(cct[test], pred_dense_test, title="dense baseline, held-out dispatches")
plt.tight_layout(); plt.show()

**What you should see.** Three regression numbers that are not the same, and a
confusion table.

The **within 50 ms of the threshold** row is the one that matters and it is
usually the worst of the three, because those are the hard cases — the ones
where the answer is genuinely close. The linear baseline scores 11.21 ms there
against 20.72 ms overall, which is the opposite pattern and happens because the
near-threshold cases sit in the middle of the range where a linear fit is best.
Say which pattern your network shows.

The **uncensored** row exists because 17 % of the labels are the number 500,
which is not a measurement of anything. A model that learns to output 500 for
those is being rewarded for predicting a stopping rule. The linear baseline
scores 16.35 ms on the uncensored subset.

In the parity plot, the orange band bottom-right holds **missed insecure cases**
and the blue band top-left holds **false alarms**. They are drawn in different
colours because they cost different things, and no single error metric in this
notebook distinguishes them. Notebook 04 is about that asymmetry.

---

## 5 · The failure that is not about accuracy

Here is the experiment the placeholder README for this exercise set asked for,
and it is the reason notebook 03 exists.

Split by **topology** instead of by dispatch: train on every contingency except
the tie outage, and test on the tie outage. No case in training has ever had
line 0 removed. The one-hot bit for branch 0 is zero in every training row.

This cell is complete, and it uses the linear baseline so that the result is a
property of the *input representation* rather than of anybody's training run.

In [ ]:
seen = cont_id != 1                    # everything except "trip line 0"
held = cont_id == 1

# Standardise on the SEEN topologies only. Reusing the statistics from the
# dispatch split would leak the held-out topology into the preprocessing, and
# would make the one-hot column for branch 0 a non-zero constant during
# training -- collinear with the bias, which produces a large and meaningless
# extrapolation instead of the real effect.
mu_t = Z[seen].mean(axis=0)
sd_t = Z[seen].std(axis=0)
sd_t = np.where(sd_t > 1e-12, sd_t, 1.0)
Zt = (Z - mu_t) / sd_t

lin_topo = ridge_fit(Zt[seen], cct[seen])
p_seen = lin_topo(Zt[seen])
p_held = lin_topo(Zt[held])

print(f"  seen topologies    MAE {np.abs(cct[seen]-p_seen).mean()*1e3:7.2f} ms"
      f"   ({seen.sum()} cases)")
print(f"  HELD-OUT topology  MAE {np.abs(cct[held]-p_held).mean()*1e3:7.2f} ms"
      f"   ({held.sum()} cases)")
print(f"  systematic bias    {(p_held-cct[held]).mean()*1e3:+7.1f} ms"
      f"   (positive = the model says the system is SAFER than it is)")
print()
c_topo = pb.confusion(cct[held], p_held)
print(f"  truly insecure held-out cases : {c_topo['tp']+c_topo['fn']}")
print(f"  of which the screen MISSED    : {c_topo['missed']}")
print(f"  recall                        : {c_topo['recall']:.3f}")

pb.plot_parity(cct[held], p_held,
               title="held-out topology: trained without ever seeing line 0 out")
plt.tight_layout(); plt.show()

**What you should see.**

```
  seen topologies    MAE   24.03 ms   (900 cases)
  HELD-OUT topology  MAE   67.42 ms   (180 cases)
  systematic bias      +67.4 ms   (positive = the model says the system is SAFER than it is)
  truly insecure held-out cases : 72
  of which the screen MISSED    : 53
  recall                        : 0.264
```

Read the bias line twice. The error is not noise — it is a **+67 ms offset in
the optimistic direction**. The model has never seen the network with the tie
removed, so it predicts what it would predict for a network with the tie
present, and that network is safer. **Fifty-three of the seventy-two genuinely
insecure cases are screened out as fine.**

This is not a training failure and no amount of data fixes it. The dense model's
only channel for "line 0 is out" is a bit that was zero in every training
example, so the corresponding weight is unconstrained. The representation, not
the optimiser, is the problem:

> **A one-hot flag names a topology. It does not describe one.** Two networks
> that differ by a single line are, to a dense model, two arbitrary and
> unrelated bit patterns. There is no sense in which one is *near* the other.

A graph network is handed the adjacency. With line 0 removed, buses 0 and 1 are
simply no longer neighbours — a change of the same kind as any other change of
neighbourhood, expressed in the same variables, and the message passing runs on
it unmodified. That is notebook 03.

---

## 6 · Save

In [ ]:
pb.save("nb02_dense",
        pred_train=pred_dense_train, pred_test=pred_dense_test,
        lin_train=lin_train, lin_test=lin_test,
        cct=cct, train=train, test=test, cont_id=cont_id, op_id=op_id,
        mu=mu, sd=sd, y_scale=np.array(Y_SCALE),
        adam=history_dense["adam"], lbfgs=history_dense["lbfgs"],
        n_params=np.array(parameter_count(dense)),
        held_out_topology_mae=np.array(np.abs(cct[held]-p_held).mean()),
        held_out_topology_recall=np.array(c_topo["recall"]))

---

## 7 · Before you move on

1. Report your dense network's test MAE against the linear baseline's 20.72 ms.
   If the gap is small, say what that tells you about how much of this problem
   is non-linear.
2. The held-out-topology bias was **positive**. Explain why an optimistic bias
   is worse than a pessimistic one of the same size, in the specific context of
   a control room deciding what to simulate properly.
3. The dense model has 42 input weights per hidden unit, one per position in the
   flattened vector. Say what happens to those weights if a seventh bus is added
   to the network, and what you would have to do to deploy the same model.
4. Section 5 used the linear baseline rather than your trained network to make
   its point. Say why that was the stronger form of the argument.

Next: **notebook 03**, where the adjacency stops being a label and becomes an
input.